<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_08-2_%EC%95%88%EC%A0%84%ED%95%9C_Tool_Contract_%EC%84%A4%EA%B3%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 지능시스템 08-2 | 안전한 Tool Contract 설계

**주제:** Input Schema · Permission · Human Approval · Actionable Error · Idempotency

**실습 목표**
1. Tool의 이름·설명·입력 구조를 명확하게 정의합니다.
2. 잘못된 입력을 **실행 전에** 차단합니다.
3. 읽기/수정/삭제 Tool의 권한과 사용자 승인을 분리합니다.
4. 오류 발생 시 **원인과 다음 행동**을 반환합니다.
5. 동일한 요청을 다시 실행해도 **중복 변경이 일어나지 않도록** 합니다.

> **안전 안내:** 모든 데이터는 이 노트북 안에 있는 **모의 데이터**입니다. 실제 문서 삭제, 이메일 발송, 결제, 외부 API 호출은 하지 않습니다. 실제 대학의 정책이나 개인정보도 사용하지 않습니다.

**실행 환경:** Google Colab / Python 3, `jsonschema` 패키지 사용

## 1. Tool은 함수뿐 아니라 계약(Contract)을 갖는다

- **Name / Description:** 모델이 언제 어떤 Tool을 써야 하는지 알 수 있게 안내
- **Input Schema:** 인자의 자료형, 필수 여부, 허용 범위 등을 정의
- **Permission / Approval:** 실제 실행을 허용할지 **프로그램**이 판단
- **Error:** 실패 원인과 사용자의 다음 행동을 설명
- **Idempotency:** 같은 작업을 다시 요청할 때 중복 실행을 막는 장치

**실행 흐름**

`Tool 요청 → Tool 존재 확인 → Schema 검증 → 권한 확인 → 사람 승인 확인 → 중복 실행 확인 → 실제 함수 실행 → 결과/오류 반환`

모델은 Tool 호출을 *제안*할 수 있지만, **권한과 안전 규칙을 결정하는 주체는 Harness(프로그램)**입니다.

## 2. 실습 환경 준비

In [1]:
# Google Colab에는 보통 jsonschema가 설치되어 있습니다.
# 아래 코드가 실행되지 않는 경우에만 다음 명령을 별도 셀에서 실행하세요.
# %pip install jsonschema

import json                       # JSON 형태의 데이터를 보기 좋게 출력
import copy                       # 모의 데이터의 깊은 복사에 사용
import hashlib                    # 승인할 요청을 구별하는 문자열 계산에 사용
from jsonschema import Draft202012Validator  # JSON Schema 입력 검사


def show(value):
    """결과 딕셔너리를 한글이 깨지지 않는 JSON 형태로 출력합니다."""
    print(json.dumps(value, ensure_ascii=False, indent=2))

print("준비 완료: 외부 API 없이 실습합니다.")

준비 완료: 외부 API 없이 실습합니다.


## 3. 모의 지식베이스와 사용자 정보 만들기

실습 중 데이터를 변경해도 실제 파일이나 데이터베이스에는 영향이 없습니다. `reset_demo()`를 실행하면 처음 상태로 되돌릴 수 있습니다.

In [2]:
# 예제 문서는 허구의 실습용 내용입니다.
INITIAL_DOCS = {
    "doc-01": {"title": "평가 안내", "text": "모의 과제 비율은 20%입니다."},
    "doc-02": {"title": "출석 안내", "text": "모의 출석 비율은 20%입니다."},
    "doc-03": {"title": "AI 사용 안내", "text": "과제 제출 시 AI 사용 내역을 표시합니다."},
}

# user_id가 서로 달라야 같은 성을 가진 사용자도 구분할 수 있습니다.
INITIAL_USERS = {
    "u001": "김학생", "u002": "김교수", "u003": "김조교"
}


def reset_demo():
    """모의 문서, 승인 기록, 중복 실행 기록을 초기화합니다."""
    global DOCS, APPROVALS, IDEMPOTENCY_RECORDS, AUDIT_LOG, NEXT_APPROVAL
    DOCS = copy.deepcopy(INITIAL_DOCS)
    APPROVALS = {}               # 승인 번호 -> 승인된 Tool/인자
    IDEMPOTENCY_RECORDS = {}     # (Tool, key) -> 이전 요청과 결과
    AUDIT_LOG = []               # 실행/거절 기록
    NEXT_APPROVAL = 1            # 실습용 승인 번호

reset_demo()
print("현재 모의 문서 수:", len(DOCS))

현재 모의 문서 수: 3


## 4. Tool의 입력 계약(JSON Schema) 정의

`query`처럼 의미가 분명한 이름을 사용합니다. `required`는 필수 입력, `additionalProperties: false`는 모르는 입력을 거부한다는 뜻입니다.

In [3]:
# 각 Tool이 받아야 하는 입력을 JSON Schema 형태로 선언합니다.
# 여기서 "type", "required" 같은 키는 JSON Schema의 표준 키워드입니다.
SEARCH_SCHEMA = {
    "type": "object",
    "properties": {
        "query": {"type": "string", "minLength": 1, "description": "검색할 단어"},
        "top_k": {"type": "integer", "minimum": 1, "maximum": 10},
    },
    "required": ["query"],
    "additionalProperties": False,
}

UPDATE_SCHEMA = {
    "type": "object",
    "properties": {
        "doc_id": {"type": "string", "minLength": 1},
        "new_text": {"type": "string", "minLength": 1},
        "idempotency_key": {"type": "string", "minLength": 1},
    },
    "required": ["doc_id", "new_text", "idempotency_key"],
    "additionalProperties": False,
}

DELETE_SCHEMA = {
    "type": "object",
    "properties": {
        "doc_id": {"type": "string", "minLength": 1},
        "idempotency_key": {"type": "string", "minLength": 1},
    },
    "required": ["doc_id", "idempotency_key"],
    "additionalProperties": False,
}

FIND_USER_SCHEMA = {
    "type": "object",
    "properties": {"name": {"type": "string", "minLength": 1}},
    "required": ["name"],
    "additionalProperties": False,
}

print("검색 Tool에 필요한 인자:", SEARCH_SCHEMA["required"])

검색 Tool에 필요한 인자: ['query']


## 5. 실제로 일을 처리하는 함수 정의

함수는 검색·수정·삭제 등의 작업을 담당합니다. **함수 자체만 믿지 않고**, 이후에 만들 Harness가 실행 전 검증을 수행합니다.

In [4]:
# 읽기 전용 Tool: 저장된 문서를 바꾸지 않고 검색 결과만 돌려줍니다.
def kb_search(query, top_k=5):
    matches = []
    for doc_id, document in DOCS.items():
        # 제목이나 내용에 검색어가 들어 있으면 결과에 추가합니다.
        if query.lower() in (document["title"] + " " + document["text"]).lower():
            matches.append({"doc_id": doc_id, **document})
    return {"results": matches[:top_k], "count": len(matches[:top_k])}


# 수정 Tool: 존재하지 않는 문서라면 실행 결과를 오류로 돌려줍니다.
def kb_update(doc_id, new_text, idempotency_key):
    if doc_id not in DOCS:
        return error_result("NOT_FOUND", "문서를 찾지 못했습니다.",
                            "먼저 kb_search로 doc_id를 확인하세요.")
    DOCS[doc_id]["text"] = new_text
    return {"status": "updated", "doc_id": doc_id, "new_text": new_text}


# 삭제 Tool: 이 실습에서만 사용하는 딕셔너리에서 항목을 제거합니다.
def kb_delete(doc_id, idempotency_key):
    if doc_id not in DOCS:
        return error_result("NOT_FOUND", "문서를 찾지 못했습니다.",
                            "삭제할 doc_id를 다시 확인하세요.")
    del DOCS[doc_id]
    return {"status": "deleted", "doc_id": doc_id}


# 사용자 검색: 같은 글자가 여러 이름에 포함되면 모호한 결과를 반환합니다.
def find_user(name):
    matches = [{"user_id": uid, "name": full_name}
               for uid, full_name in INITIAL_USERS.items() if name in full_name]
    if not matches:
        return error_result("NOT_FOUND", "일치하는 사용자가 없습니다.",
                            "다른 이름으로 다시 검색하세요.")
    if len(matches) > 1:
        return error_result("AMBIGUOUS", "일치하는 사용자가 여러 명입니다.",
                            "후보 중 정확한 사용자 ID를 확인하세요.", candidates=matches)
    return {"user": matches[0]}


# 실패 이유를 파악할 수 있도록 공통 오류 형식을 만듭니다.
# Python에서는 호출 시점에 함수 이름을 찾으므로 아래에 정의해도 괜찮습니다.
def error_result(code, message, next_action, **extra):
    return {"ok": False, "error_code": code, "message": message,
            "next_action": next_action, **extra}

print("모의 Tool 함수 4개 정의 완료")

모의 Tool 함수 4개 정의 완료


## 6. Tool Registry: 이름·설명·권한·실행 함수 등록

`requires_approval`이 `True`이면 승인 없이 실행하지 않습니다. `allowed_roles`는 Tool 사용을 허용하는 역할입니다. 이는 모델에게 쓰는 문장이 아니라 **프로그램이 검사할 정책 데이터**입니다.

In [5]:
TOOLS = {
    "kb_search": {
        "description": "모의 강의 정책 문서를 검색한다. 문서를 변경하지 않는다.",
        "schema": SEARCH_SCHEMA, "fn": kb_search,
        "read_only": True, "destructive": False,
        "requires_approval": False, "allowed_roles": ["student", "teacher"],
    },
    "find_user": {
        "description": "이름으로 사용자 후보를 검색한다. 여러 명이면 후보를 돌려준다.",
        "schema": FIND_USER_SCHEMA, "fn": find_user,
        "read_only": True, "destructive": False,
        "requires_approval": False, "allowed_roles": ["student", "teacher"],
    },
    "kb_update": {
        "description": "문서 본문을 수정한다. 교사 역할과 사람의 승인이 필요하다.",
        "schema": UPDATE_SCHEMA, "fn": kb_update,
        "read_only": False, "destructive": False,
        "requires_approval": True, "allowed_roles": ["teacher"],
    },
    "kb_delete": {
        "description": "문서를 삭제한다. 교사 역할과 사람의 승인이 필요하다.",
        "schema": DELETE_SCHEMA, "fn": kb_delete,
        "read_only": False, "destructive": True,
        "requires_approval": True, "allowed_roles": ["teacher"],
    },
}

for name, tool in TOOLS.items():
    print(f"{name:10s} | 읽기 전용: {tool['read_only']} | 승인 필요: {tool['requires_approval']}")

kb_search  | 읽기 전용: True | 승인 필요: False
find_user  | 읽기 전용: True | 승인 필요: False
kb_update  | 읽기 전용: False | 승인 필요: True
kb_delete  | 읽기 전용: False | 승인 필요: True


## 7. Schema 검증: 잘못된 인자는 함수를 부르기 전에 막기

다음 코드는 `jsonschema`로 **필수 필드 누락, 범위 초과, 잘못된 타입, 허용되지 않은 필드**를 검사합니다. 이 단계에서는 아직 문서를 수정하거나 삭제하지 않습니다.

In [6]:
def validate_args(schema, args):
    """입력이 Schema에 맞는지 확인해 (성공 여부, 오류 메시지)를 반환합니다."""
    if not isinstance(args, dict):
        return False, "Tool 인자는 JSON 객체(dict)여야 합니다."
    validator = Draft202012Validator(schema)
    # 가능한 오류를 정렬해 첫 번째 오류부터 학생에게 보여줍니다.
    problems = sorted(validator.iter_errors(args), key=lambda e: (list(map(str, e.path)), e.message))
    if problems:
        return False, problems[0].message
    return True, "valid"


# 실패해야 하는 입력 4개를 시험합니다.
test_inputs = [
    {},                                # query가 없습니다.
    {"query": "출석", "top_k": 99},    # 상한 10을 초과합니다.
    {"query": "출석", "top_k": "5"},   # 정수가 아니라 문자열입니다.
    {"query": "출석", "unknown": 1},   # 정의되지 않은 필드입니다.
    {"query": "출석", "top_k": 2},      # 올바른 입력입니다.
]
for args in test_inputs:
    valid, message = validate_args(SEARCH_SCHEMA, args)
    print(f"입력={args} → 유효={valid}, 이유={message}")

입력={} → 유효=False, 이유='query' is a required property
입력={'query': '출석', 'top_k': 99} → 유효=False, 이유=99 is greater than the maximum of 10
입력={'query': '출석', 'top_k': '5'} → 유효=False, 이유='5' is not of type 'integer'
입력={'query': '출석', 'unknown': 1} → 유효=False, 이유=Additional properties are not allowed ('unknown' was unexpected)
입력={'query': '출석', 'top_k': 2} → 유효=True, 이유=valid


## 8. Human Approval: 승인한 요청과 실제 실행 요청을 연결하기

단순히 `approved=True`만 전달하면 **무엇을 승인했는지** 알 수 없습니다. 따라서 이 실습에서는 승인 번호를 발급할 때 **Tool 이름과 입력 인자**를 함께 기록합니다. 요청이 달라지면 이전 승인을 사용할 수 없습니다.

> 실제 서비스에서는 승인자의 신원, 인증, 만료 시각, 위·변조 방지, 저장소 등을 추가해야 합니다. 여기서는 학습을 위해 메모리에만 저장합니다.

In [7]:
def request_signature(tool_name, args):
    """동일한 Tool 이름/인자는 동일한 서명이 나오도록 합니다."""
    # sort_keys=True: 딕셔너리의 키 순서가 달라도 같은 문자열 생성
    payload = json.dumps({"tool": tool_name, "args": args},
                         ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def grant_approval(tool_name, args, approver="담당자"):
    """교수자 또는 승인 UI가 수행한다고 가정하는 '모의 승인' 함수입니다."""
    global NEXT_APPROVAL
    approval_id = f"approval-{NEXT_APPROVAL:03d}"
    NEXT_APPROVAL += 1
    APPROVALS[approval_id] = {
        "signature": request_signature(tool_name, args),
        "approver": approver,
        "used": False,  # 승인 번호의 재사용을 막기 위한 표시
    }
    return approval_id


# 승인은 의도한 요청에만 사용 가능합니다.
example_args = {"doc_id": "doc-01", "new_text": "수정 예시", "idempotency_key": "demo-001"}
example_approval = grant_approval("kb_update", example_args)
print("승인 ID:", example_approval)
print("승인된 요청과 일치:", APPROVALS[example_approval]["signature"] ==
      request_signature("kb_update", example_args))

승인 ID: approval-001
승인된 요청과 일치: True


## 9. 핵심 함수: Harness가 안전 규칙을 검사한 뒤 Tool 실행

- `role`: 요청자의 역할 (`student` 또는 `teacher`)
- `approval_id`: 승인된 변경/삭제 요청에 붙이는 번호
- `idempotency_key`: 한 번 성공한 변경/삭제 요청의 중복 실행을 방지하는 키

**중요:** 모델이 만들어 낸 요청이라도, `call_tool()`의 검사를 통과해야만 실행됩니다. 재시도한 요청은 **이미 처리된 성공 결과**를 다시 확인할 수 있습니다.

In [8]:
def call_tool(tool_name, args, *, role="student", approval_id=None):
    """Tool 호출을 위한 미니 Harness: 검증 → 권한 → 승인 → 중복 확인 → 실행."""

    # 1. 실제로 존재하는 Tool인지 확인합니다.
    tool = TOOLS.get(tool_name)
    if tool is None:
        result = error_result("UNKNOWN_TOOL", "등록되지 않은 Tool입니다.",
                              "TOOLS에 등록된 이름 중에서 선택하세요.")
        AUDIT_LOG.append({"tool": tool_name, "role": role, "status": "blocked", "reason": "UNKNOWN_TOOL"})
        return result

    # 2. 실행하기 전에 입력값이 Schema를 만족하는지 확인합니다.
    valid, message = validate_args(tool["schema"], args)
    if not valid:
        result = error_result("INVALID_ARGUMENT", f"입력 검증 실패: {message}",
                              "필수 인자, 자료형, 허용 범위를 확인하고 수정하세요.")
        AUDIT_LOG.append({"tool": tool_name, "role": role, "status": "blocked", "reason": "INVALID_ARGUMENT"})
        return result

    # 3. 요청자가 이 Tool을 사용할 수 있는 역할인지 확인합니다.
    if role not in tool["allowed_roles"]:
        result = error_result("PERMISSION_DENIED", "이 역할은 해당 Tool을 사용할 수 없습니다.",
                              "허용된 사용자 권한을 확인하세요. 같은 입력을 반복해도 해결되지 않습니다.")
        AUDIT_LOG.append({"tool": tool_name, "role": role, "status": "blocked", "reason": "PERMISSION_DENIED"})
        return result

    # 4. 승인 대상 Tool은 '정확히 이 요청'을 승인받았는지 확인합니다.
    if tool["requires_approval"]:
        approval = APPROVALS.get(approval_id)
        if (approval is None or approval["used"] or
                approval["signature"] != request_signature(tool_name, args)):
            result = error_result("APPROVAL_REQUIRED", "이 실행 요청에 대한 유효한 승인이 없습니다.",
                                  "변경할 Tool과 인자를 확인하고 별도로 승인받으세요.")
            AUDIT_LOG.append({"tool": tool_name, "role": role, "status": "blocked", "reason": "APPROVAL_REQUIRED"})
            return result

    # 5. 승인 검사를 통과한 뒤 승인 번호를 1회 사용 처리합니다.
    # 실제 시스템에서는 승인 소비와 실행 기록이 트랜잭션으로 관리되어야 합니다.
    if tool["requires_approval"]:
        APPROVALS[approval_id]["used"] = True

    # 6. 동일 요청의 중복 실행을 방지합니다. (쓰기 Tool만 대상)
    idem_key = args.get("idempotency_key") if not tool["read_only"] else None
    record_key = (tool_name, idem_key) if idem_key else None
    signature = request_signature(tool_name, args)
    if record_key is not None and record_key in IDEMPOTENCY_RECORDS:
        previous = IDEMPOTENCY_RECORDS[record_key]
        if previous["signature"] != signature:
            result = error_result("IDEMPOTENCY_CONFLICT", "같은 키에 다른 입력을 사용했습니다.",
                                  "다른 작업에는 새로운 idempotency_key를 사용하세요.")
            AUDIT_LOG.append({"tool": tool_name, "role": role, "status": "blocked", "reason": "IDEMPOTENCY_CONFLICT"})
            return result
        AUDIT_LOG.append({"tool": tool_name, "role": role, "status": "replayed"})
        return {"ok": True, "replayed": True, "data": copy.deepcopy(previous["data"])}

    # 7. 실제 함수를 호출합니다. 앞의 검사를 거쳐야 이 줄이 실행됩니다.
    data = tool["fn"](**args)
    if isinstance(data, dict) and data.get("ok") is False:
        AUDIT_LOG.append({"tool": tool_name, "role": role, "status": "error", "reason": data["error_code"]})
        return data

    # 8. 성공한 쓰기 작업만 저장합니다. 재요청 시 같은 결과를 돌려줍니다.
    if record_key is not None:
        IDEMPOTENCY_RECORDS[record_key] = {"signature": signature, "data": copy.deepcopy(data)}
    AUDIT_LOG.append({"tool": tool_name, "role": role, "status": "executed"})
    return {"ok": True, "replayed": False, "data": data}

print("Harness 준비 완료")

Harness 준비 완료


## 10. 실습 A — 읽기 Tool과 잘못된 입력

In [9]:
# 학생 권한으로는 읽기 Tool을 정상 사용할 수 있습니다.
print("[정상 검색]")
show(call_tool("kb_search", {"query": "출석", "top_k": 2}, role="student"))

# top_k가 최대 10을 넘으면 검색 함수가 실행되기 전에 거절됩니다.
print("\n[잘못된 인자]")
show(call_tool("kb_search", {"query": "출석", "top_k": 100}, role="student"))

# 사용자 후보가 여러 명이면 모호함을 설명하는 오류를 반환합니다.
print("\n[후보가 여러 명]")
show(call_tool("find_user", {"name": "김"}, role="student"))

[정상 검색]
{
  "ok": true,
  "replayed": false,
  "data": {
    "results": [
      {
        "doc_id": "doc-02",
        "title": "출석 안내",
        "text": "모의 출석 비율은 20%입니다."
      }
    ],
    "count": 1
  }
}

[잘못된 인자]
{
  "ok": false,
  "error_code": "INVALID_ARGUMENT",
  "message": "입력 검증 실패: 100 is greater than the maximum of 10",
  "next_action": "필수 인자, 자료형, 허용 범위를 확인하고 수정하세요."
}

[후보가 여러 명]
{
  "ok": false,
  "error_code": "AMBIGUOUS",
  "message": "일치하는 사용자가 여러 명입니다.",
  "next_action": "후보 중 정확한 사용자 ID를 확인하세요.",
  "candidates": [
    {
      "user_id": "u001",
      "name": "김학생"
    },
    {
      "user_id": "u002",
      "name": "김교수"
    },
    {
      "user_id": "u003",
      "name": "김조교"
    }
  ]
}


## 11. 실습 B — 권한 없는 수정과 승인 없는 수정 차단

In [10]:
change_args = {
    "doc_id": "doc-01", "new_text": "새로운 모의 평가 안내입니다.",
    "idempotency_key": "run-001-update-01"
}

print("[학생이 수정 시도 → 권한 없음]")
show(call_tool("kb_update", change_args, role="student"))

print("\n[교사가 승인 없이 수정 시도 → 승인 필요]")
show(call_tool("kb_update", change_args, role="teacher"))

print("\n[차단 후에도 원본 내용이 그대로인지 확인]")
print(DOCS["doc-01"]["text"])

[학생이 수정 시도 → 권한 없음]
{
  "ok": false,
  "error_code": "PERMISSION_DENIED",
  "message": "이 역할은 해당 Tool을 사용할 수 없습니다.",
  "next_action": "허용된 사용자 권한을 확인하세요. 같은 입력을 반복해도 해결되지 않습니다."
}

[교사가 승인 없이 수정 시도 → 승인 필요]
{
  "ok": false,
  "error_code": "APPROVAL_REQUIRED",
  "message": "이 실행 요청에 대한 유효한 승인이 없습니다.",
  "next_action": "변경할 Tool과 인자를 확인하고 별도로 승인받으세요."
}

[차단 후에도 원본 내용이 그대로인지 확인]
모의 과제 비율은 20%입니다.


## 12. 실습 C — 같은 요청에 대해서만 승인받아 수정하기

In [11]:
# 1. 요청과 인자를 확인한 뒤 모의 승인 번호를 발급합니다.
change_approval = grant_approval("kb_update", change_args, approver="실습 승인자")

# 2. 승인 번호를 붙여서 정확히 같은 Tool, 같은 인자로 실행합니다.
print("[승인된 문서 수정]")
show(call_tool("kb_update", change_args, role="teacher", approval_id=change_approval))
print("수정 후 문서 내용:", DOCS["doc-01"]["text"])

# 3. 이미 사용한 승인 번호로 다른 수정 작업을 요청하면 거절됩니다.
print("\n[승인 번호 재사용 시도]")
show(call_tool("kb_update", {**change_args, "new_text": "승인받지 않은 문장"},
               role="teacher", approval_id=change_approval))

[승인된 문서 수정]
{
  "ok": true,
  "replayed": false,
  "data": {
    "status": "updated",
    "doc_id": "doc-01",
    "new_text": "새로운 모의 평가 안내입니다."
  }
}
수정 후 문서 내용: 새로운 모의 평가 안내입니다.

[승인 번호 재사용 시도]
{
  "ok": false,
  "error_code": "APPROVAL_REQUIRED",
  "message": "이 실행 요청에 대한 유효한 승인이 없습니다.",
  "next_action": "변경할 Tool과 인자를 확인하고 별도로 승인받으세요."
}


## 13. 실습 D — Idempotency로 중복 수정을 막기

같은 작업을 재시도할 때 새 승인 번호를 발급받아도, **idempotency_key**가 이전 성공 요청과 같다면 실제 수정 함수를 다시 실행하지 않습니다. 단, 이 노트북은 교육용 메모리 저장소이므로 런타임이 초기화되면 기록이 사라집니다.

In [12]:
# 1. 같은 Tool과 같은 인자에 대한 새로운 승인 번호를 생성합니다.
retry_approval = grant_approval("kb_update", change_args)

# 2. 이전과 같은 요청을 다시 보내도 이전 성공 결과를 반환합니다.
retry_result = call_tool("kb_update", change_args, role="teacher", approval_id=retry_approval)
print("[같은 요청 재시도]")
show(retry_result)
print("실제 함수 재실행 여부:", "재실행 안 함" if retry_result.get("replayed") else "실행함")

# 3. 같은 키를 다른 본문에 재사용하면 충돌 오류가 발생합니다.
other_args = {**change_args, "new_text": "다른 내용을 덮어쓸까요?"}
other_approval = grant_approval("kb_update", other_args)
print("\n[같은 키로 다른 내용 요청]")
show(call_tool("kb_update", other_args, role="teacher", approval_id=other_approval))

[같은 요청 재시도]
{
  "ok": true,
  "replayed": true,
  "data": {
    "status": "updated",
    "doc_id": "doc-01",
    "new_text": "새로운 모의 평가 안내입니다."
  }
}
실제 함수 재실행 여부: 재실행 안 함

[같은 키로 다른 내용 요청]
{
  "ok": false,
  "error_code": "IDEMPOTENCY_CONFLICT",
  "message": "같은 키에 다른 입력을 사용했습니다.",
  "next_action": "다른 작업에는 새로운 idempotency_key를 사용하세요."
}


## 14. 실습 E — 삭제는 권한과 승인이 모두 필요하다

In [13]:
# 삭제도 실제 외부 데이터가 아니라 DOCS 딕셔너리에서만 실행됩니다.
delete_args = {"doc_id": "doc-02", "idempotency_key": "run-001-delete-02"}

# 먼저 승인 없이 삭제를 요청합니다.
print("[승인 없는 삭제]")
show(call_tool("kb_delete", delete_args, role="teacher"))
print("doc-02 존재 여부:", "doc-02" in DOCS)

# 정확한 삭제 요청을 승인받은 다음 다시 호출합니다.
delete_approval = grant_approval("kb_delete", delete_args)
print("\n[승인된 삭제]")
show(call_tool("kb_delete", delete_args, role="teacher", approval_id=delete_approval))
print("doc-02 존재 여부:", "doc-02" in DOCS)

[승인 없는 삭제]
{
  "ok": false,
  "error_code": "APPROVAL_REQUIRED",
  "message": "이 실행 요청에 대한 유효한 승인이 없습니다.",
  "next_action": "변경할 Tool과 인자를 확인하고 별도로 승인받으세요."
}
doc-02 존재 여부: True

[승인된 삭제]
{
  "ok": true,
  "replayed": false,
  "data": {
    "status": "deleted",
    "doc_id": "doc-02"
  }
}
doc-02 존재 여부: False


## 15. 실행 기록(Audit Log) 확인

실행된 호출뿐 아니라 **차단된 호출**도 기록해야 정책이 잘 작동했는지 분석할 수 있습니다. 실제 운영에서는 민감한 입력을 그대로 로그에 기록하지 않도록 주의합니다.

In [14]:
# 앞에서 실행한 요청들의 결과 상태를 한눈에 봅니다.
for number, event in enumerate(AUDIT_LOG, start=1):
    print(f"{number:02d}. Tool={event['tool']:<12} 역할={event['role']:<8} "
          f"상태={event['status']:<9} 이유={event.get('reason', '-')}")

01. Tool=kb_search    역할=student  상태=executed  이유=-
02. Tool=kb_search    역할=student  상태=blocked   이유=INVALID_ARGUMENT
03. Tool=find_user    역할=student  상태=error     이유=AMBIGUOUS
04. Tool=kb_update    역할=student  상태=blocked   이유=PERMISSION_DENIED
05. Tool=kb_update    역할=teacher  상태=blocked   이유=APPROVAL_REQUIRED
06. Tool=kb_update    역할=teacher  상태=executed  이유=-
07. Tool=kb_update    역할=teacher  상태=blocked   이유=APPROVAL_REQUIRED
08. Tool=kb_update    역할=teacher  상태=replayed  이유=-
09. Tool=kb_update    역할=teacher  상태=blocked   이유=IDEMPOTENCY_CONFLICT
10. Tool=kb_delete    역할=teacher  상태=blocked   이유=APPROVAL_REQUIRED
11. Tool=kb_delete    역할=teacher  상태=executed  이유=-


## 16. 자동 테스트: 실습 완료 여부 확인

테스트 전에 실습 데이터를 초기화하여, **셀을 여러 번 실행해도 같은 결과**가 나오도록 합니다. 다음 7가지 검사를 모두 통과해야 합니다.

In [15]:
# 재현 가능한 테스트를 위해 모의 데이터와 승인 기록을 초기화합니다.
reset_demo()

# ① 올바른 검색 입력은 성공해야 합니다.
r1 = call_tool("kb_search", {"query": "출석", "top_k": 2})
assert r1["ok"] is True and r1["data"]["count"] == 1

# ② 잘못된 인자는 Tool 실행 전에 차단되어야 합니다.
r2 = call_tool("kb_search", {"query": "출석", "top_k": 99})
assert r2["error_code"] == "INVALID_ARGUMENT"

# ③ 학생이 쓰기 Tool을 쓰면 거절되어야 합니다.
a = {"doc_id": "doc-01", "new_text": "승인된 수정", "idempotency_key": "test-001"}
r3 = call_tool("kb_update", a, role="student")
assert r3["error_code"] == "PERMISSION_DENIED"

# ④ 교사라 해도 승인 없이는 수정하지 못해야 합니다.
r4 = call_tool("kb_update", a, role="teacher")
assert r4["error_code"] == "APPROVAL_REQUIRED"
assert DOCS["doc-01"]["text"] == INITIAL_DOCS["doc-01"]["text"]

# ⑤ 승인된 요청은 성공해야 합니다.
approval = grant_approval("kb_update", a)
r5 = call_tool("kb_update", a, role="teacher", approval_id=approval)
assert r5["ok"] is True and DOCS["doc-01"]["text"] == "승인된 수정"

# ⑥ 같은 키 + 같은 입력 재요청은 중복 실행 없이 이전 결과를 돌려줍니다.
approval2 = grant_approval("kb_update", a)
r6 = call_tool("kb_update", a, role="teacher", approval_id=approval2)
assert r6["ok"] is True and r6["replayed"] is True

# ⑦ 같은 키 + 다른 내용은 거절되어야 합니다.
b = {**a, "new_text": "다른 수정"}
approval3 = grant_approval("kb_update", b)
r7 = call_tool("kb_update", b, role="teacher", approval_id=approval3)
assert r7["error_code"] == "IDEMPOTENCY_CONFLICT"
assert DOCS["doc-01"]["text"] == "승인된 수정"

print("✅ 자동 테스트 7개 모두 통과했습니다.")

✅ 자동 테스트 7개 모두 통과했습니다.


### 실제 서비스로 확장할 때의 주의점
- 이 실습의 `grant_approval()`은 **모의 승인 함수**이며, 실제 사용자 인증을 수행하지 않습니다.
- 인메모리 멱등성 저장소는 프로세스 종료·동시 실행에 안전하지 않습니다. 실제 시스템에서는 영속 저장소와 원자적 처리 등이 필요합니다.
- `read_only`, `destructive` 같은 메타데이터는 Tool의 **계약 선언**입니다. 실제 부작용과 일치하도록 관리해야 합니다.
